# Kobol Expander VCO: grey-box model, full pipeline
Run top to bottom in Colab (CPU is enough, about 2 h for the full cross-validation; set `STEPS` lower for a quick check).
Upload this folder's scripts next to the notebook, or clone the repo.

In [ ]:
!pip -q install soundfile torch scipy pandas h5py matplotlib
!curl -sL -o kobol.zip https://www.kaggle.com/api/v1/datasets/download/bringmethetxcos/kobolrsf-vco && unzip -qo kobol.zip -d kobol
# thesis model for the pitch baseline
!mkdir -p KobolRSF/models && curl -sL -o KobolRSF/models/PU.h5 https://github.com/sofiavallejob/KobolRSF/raw/main/models/PU.h5
STEPS=1500  # per fold; 1500 was used for the reported numbers

## 1. Re-estimate f0 and extract harmonics
ACF coarse estimate, FFT refinement. 256 harmonics per file, phase aligned to the fundamental, masked above 18 kHz.

In [ ]:
!python f0refine.py
!python extract.py

## 2. Pitch: physical model, ablations, baselines, three CV schemes

In [ ]:
!python pitch_cv.py
!python pitch_nooff.py
!python pitchphys2.py   # full-data fit used for deployment and figures

## 3. Waveform: cross-validation of all variants
Schemes: rand5, col (waveform columns held out), row (CV rows held out). Saves predictions in runs/.

In [ ]:
import json, subprocess
plan=[['rand5','final',STEPS],['rand5','bb',4000],['col','final',STEPS],['col','bb',4000],['row','final',STEPS],['row','bb',4000],
      ['rand5','full',STEPS],['rand5','-tri',STEPS],['rand5','-compB',STEPS],['rand5','-lowpass',STEPS],['rand5','-chainHP',STEPS],
      ['rand5','+slew',STEPS],['rand5','+lp2',STEPS],['rand5','+reset',STEPS],['rand5','l1',STEPS],['rand5','final',STEPS,1],['rand5','final',STEPS,2]]
subprocess.run(['python','exp.py',json.dumps(plan)],check=True)

## 4. Analysis: medians, bootstrap CIs, families, registers; raw-WAV evaluation

In [ ]:
!python analyze.py
!python evalraw.py rand5 final

## 5. Final model on all data, export for the browser, figures

In [ ]:
!python trainfinal.py 3000
!python figpitch.py && python figparams.py && python figcv.py && python figcyc.py